Basado en el notebook de la clase (autor: **Rodrigo López Vera — Data Science, Revolut Perú**)

---

Para setear la key antes de abrir Jupyter:
```bash
export GOOGLE_API_KEY=AIza...
jupyter notebook  # o jupyter lab
```

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import json
import os
import subprocess
import time
from pathlib import Path

import httpx

# ── Rutas ─────────────────────────────────────────────────────────────────────
# Este notebook vive en session_1/. El directorio de trabajo del kernel
# es el mismo directorio donde está el notebook.
SESSION_DIR = Path.cwd()          # .../modelos_en_produccion/session_1
RAG_API_DIR = SESSION_DIR / "rag-api"

In [ ]:

assert RAG_API_DIR.exists(), (
    f"No se encontró rag-api/ en {SESSION_DIR}.\n"
    "Asegúrate de abrir Jupyter desde session_1/."
)

# ── Preflight: verificar herramientas ─────────────────────────────────────────
def check_tool(name):
    r = subprocess.run([name, "--version"], capture_output=True, text=True)
    if r.returncode == 0:
        version = r.stdout.strip().splitlines()[0]
        print(f"  OK  {name}: {version}")
    else:
        print(f"  FALTA  {name} — instálalo antes de continuar")

print("Verificando herramientas:")
check_tool("docker")
check_tool("uv")
print()

# ── Verificar que Docker Desktop está corriendo ────────────────────────────────
r = subprocess.run(["docker", "info"], capture_output=True, text=True)
if r.returncode == 0:
    print("  OK  Docker Desktop está corriendo")
else:
    print("  ERROR  Docker Desktop no responde — ábrelo antes de continuar")
print()

# ── Verificar GOOGLE_API_KEY ───────────────────────────────────────────────────
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
if GOOGLE_API_KEY:
    print(f"  OK  GOOGLE_API_KEY encontrada ({GOOGLE_API_KEY[:8]}...)")
else:
    print("  AVISO  GOOGLE_API_KEY no está en el entorno.")
    print("         Necesitarás setearla antes del Bloque 04.")
    print("         Cierra Jupyter, ejecuta: export GOOGLE_API_KEY=AIza...")
    print("         Y vuelve a abrir Jupyter desde esa misma terminal.")
print()

# ── Variable global para el ID del contenedor (se llena en Bloque 04) ─────────
CONTAINER_ID = None

print(f"SESSION_DIR : {SESSION_DIR}")
print(f"RAG_API_DIR : {RAG_API_DIR}")
print("Setup completo.")

In [ ]:
# Correr uv lock para ver el output en vivo
result = subprocess.run(
    ["uv", "lock"],
    cwd=RAG_API_DIR,
    capture_output=True,
    text=True
)
print(result.stdout or "(sin output — lockfile ya está al día)")
if result.returncode != 0:
    print("STDERR:", result.stderr)

In [ ]:
print("Iniciando build... (puede tardar 3-8 min la primera vez)")
print(f"Directorio: {RAG_API_DIR}")
print()

start = time.time()
result = subprocess.run(
    ["docker", "build", "-t", "rag-compliance-api", "."],
    cwd=RAG_API_DIR,
    capture_output=True,
    text=True
)
elapsed = time.time() - start

if result.returncode == 0:
    print(f"Build exitoso en {elapsed:.1f}s")
    print()
    # Mostrar solo las líneas relevantes (capas del build)
    for line in result.stderr.splitlines():
        if any(kw in line for kw in ["CACHED", "RUN", "COPY", "FROM", "CMD", "=>", "Successfully"]):
            print(line)
else:
    print(f"Build fallido después de {elapsed:.1f}s")
    print(result.stderr[-2000:])
    print("--- STDERR ---")
    print(result.stderr[-2000:])

In [ ]:
# Tamaño y capas de la imagen construida
r = subprocess.run(["docker", "images", "rag-compliance-api"], capture_output=True, text=True)
print(r.stdout)

# Historial de capas con sus tamaños
r2 = subprocess.run(
    ["docker", "history", "rag-compliance-api", "--format", "table {{.CreatedBy}}\t{{.Size}}"],
    capture_output=True, text=True
)
print(r2.stdout)

In [ ]:
if not GOOGLE_API_KEY:
    print("ERROR: GOOGLE_API_KEY no está en el entorno.")
    print("Ejecuta en tu terminal: export GOOGLE_API_KEY=AIza...")
    print("Luego reinicia el kernel y vuelve a correr desde la celda de Setup.")
else:
    r = subprocess.run(
        ["docker", "run", "-d", "-p", "8000:8000",
         "-e", f"GOOGLE_API_KEY={GOOGLE_API_KEY}",
         "rag-compliance-api"],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        CONTAINER_ID = r.stdout.strip()
        print(f"Contenedor arrancado: {CONTAINER_ID[:12]}")
        print()
        print("Equivalente en terminal:")
        print(f"  docker run -d -p 8000:8000 -e GOOGLE_API_KEY=AIza... rag-compliance-api")
    else:
        print("Error al arrancar el contenedor:")
        print(r.stderr)

In [ ]:
# Esperar a que la app esté lista (máx 45 segundos)
BASE = "http://localhost:8000"
print("Esperando que la API levante...", end="")

for i in range(45):
    try:
        r = httpx.get(f"{BASE}/", timeout=2)
        if r.status_code == 200:
            print(f" lista en {i+1}s")
            print(json.dumps(r.json(), indent=2, ensure_ascii=False))
            break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(1)
else:
    print(" timeout")
    if CONTAINER_ID:
        r_logs = subprocess.run(["docker", "logs", CONTAINER_ID], capture_output=True, text=True)
        print("\nLogs del contenedor:")
        print(r_logs.stdout[-2000:])
        print(r_logs.stderr[-2000:])

In [ ]:
# Ingestar una circular SBS
# ChromaDB arranca vacío en cada contenedor nuevo — siempre hay que ingestar
json_path = RAG_API_DIR / "data" / "ddl.json"

with open(json_path, "rb") as f:
    r = httpx.post(
        f"{BASE}/ingest",
        files={"file": (json_path.name, f, "application/json")},
        timeout=30,
    )

print("POST /ingest →", r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

In [ ]:
# Query
r = httpx.post(
    f"{BASE}/query/json",
    json={"question": "¿Qué productos hay disponibles?"},
    timeout=30,
)
print("POST /query/json →", r.status_code)
resp = r.json()
print("\nQuery respuesta:", resp.get("answer", ""))
print("\nTabla:", resp.get("sources", []))

Prueba libre

In [ ]:
# Query
question = input('Ingrese su consulta')
r = httpx.post(
    f"{BASE}/query/json",
    json={f"question": {question}},
    timeout=30,
)
print("POST /query/json →", r.status_code)
resp = r.json()
print("\nQuery respuesta:", resp.get("answer", ""))
print("\nTabla:", resp.get("sources", []))

In [ ]:
# Detener y eliminar el contenedor al final de la sesión
if CONTAINER_ID:
    subprocess.run(["docker", "stop", CONTAINER_ID], capture_output=True)
    subprocess.run(["docker", "rm", CONTAINER_ID], capture_output=True)
    print(f"Contenedor {CONTAINER_ID[:12]} detenido y eliminado.")
    CONTAINER_ID = None
else:
    print("No hay contenedor activo.")

# Listar imágenes (la imagen queda — la usamos en sesión 2)
r = subprocess.run(["docker", "images", "rag-compliance-api"], capture_output=True, text=True)
print(r.stdout)